# SIFE-LDM V2 — Colab GPU-Accelerated Training Notebook

**Architecture:** Complex-field diffusion with Phase-Routed MoE, Token Merging, Learnable Meta-Physics, Continuous-time SDE, and Phase-Coherent CFG conditioning.

**Dataset:** CIFAR-100 (100-class, 32×32 images)

**GPU:** Configured for **A100 / V100 / T4** Colab GPUs with automatic VRAM-based scaling.

**Setup Modes:**
- **Antigravity Extension** — files auto-synced from your local workspace
- **Google Drive** — upload the project folder to Drive and mount it
- **Direct Upload** — upload a zip of the project to Colab

**Outputs:** Class-conditional image generation via `cfg_guided_sample()`

## Cell 1: Workspace Setup & Environment
Detects whether you're using the Antigravity Extension, Google Drive, or a direct upload, and sets up the working directory accordingly.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

# --- Workspace Detection ---
WORKING_DIR = '/content/sife-ldm'

def find_project_root(search_dir):
    """Walk a directory tree looking for our project marker files."""
    for root, dirs, files in os.walk(search_dir):
        if 'train_vision.py' in files or (Path(root) / 'scripts' / 'train_vision.py').exists():
            # found the project root (one level up from scripts/)
            return root if 'sife' in dirs else str(Path(root).parent)
    return None

# Mode 1: Antigravity Extension — files are already synced to /content
if os.path.exists('/content/sife') and os.path.exists('/content/scripts/train_vision.py'):
    WORKING_DIR = '/content'
    print('✅  Antigravity Extension detected — workspace already synced.')

# Mode 2: Google Drive mount
elif os.path.exists('/content/drive'):
    print('🔍  Checking Google Drive for project...')
    drive_root = find_project_root('/content/drive/MyDrive')
    if drive_root:
        print(f'📁  Found project at: {drive_root}')
        os.makedirs(WORKING_DIR, exist_ok=True)
        !cp -rf {drive_root}/* {WORKING_DIR}/
        print('✅  Project copied from Google Drive.')
    else:
        print('⚠️  Project not found in Drive. Trying to mount...')
        from google.colab import drive
        drive.mount('/content/drive')
        drive_root = find_project_root('/content/drive/MyDrive')
        if drive_root:
            os.makedirs(WORKING_DIR, exist_ok=True)
            !cp -rf {drive_root}/* {WORKING_DIR}/
            print('✅  Project copied from Google Drive after mounting.')
        else:
            print('❌  Project not found in Drive.')
            print('    Upload your project folder to Google Drive/sife-ldm/ and re-run.')

# Mode 3: Direct upload — look for uploaded zip or folder in /content
elif os.path.exists('/content'):
    # Check for uploaded zip file
    zips = list(Path('/content').glob('sife*.zip'))
    if zips:
        print(f'📦  Found uploaded zip: {zips[0]}')
        os.makedirs(WORKING_DIR, exist_ok=True)
        !unzip -qo {zips[0]} -d {WORKING_DIR}
        print('✅  Project extracted from zip.')
    else:
        # Check if project was uploaded as a folder
        uploaded_root = find_project_root('/content')
        if uploaded_root:
            WORKING_DIR = uploaded_root
            print(f'✅  Found uploaded project at: {WORKING_DIR}')
        else:
            print('❌  No project files found!')
            print('    Option A: Use Antigravity Extension to sync files automatically.')
            print('    Option B: Upload sife-ldm.zip to /content/ via the Files panel.')
            print('    Option C: Mount Google Drive with the project folder.')

# Change to working directory
%cd {WORKING_DIR}
print(f'\n📂  Working directory: {os.getcwd()}')
print(f'📄  Files: {os.listdir(".")[:10]}...')

# Verify critical files exist
critical = ['sife/model.py', 'sife/diffusion.py', 'scripts/train_vision.py']
missing = [f for f in critical if not os.path.exists(f)]
if missing:
    print(f'\n❌  Missing critical files: {missing}')
else:
    print('\n✅  All critical project files present.')

## Cell 2: GPU Detection & Auto-Scaling
Detects your Colab GPU and automatically configures optimal training parameters.

In [ ]:
import os, sys, json

# Prevent JAX from pre-allocating all GPU memory
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'

import jax
import jax.numpy as jnp

print(f'Python: {sys.version}')
print(f'JAX:    {jax.__version__}')
print(f'Devices: {jax.devices()}')

# --- GPU Detection & Auto-Scaling ---
def detect_gpu_and_scale():
    """Detect Colab GPU type and return optimal training params."""
    devices = jax.devices()
    gpu_device = None
    for d in devices:
        if d.platform == 'gpu':
            gpu_device = d
            break

    if gpu_device is None:
        print('⚠️  No GPU detected! Training will be extremely slow.')
        print('    Go to Runtime → Change runtime type → GPU')
        return {'batch_size': 4, 'embed_dim': 64, 'base_features': 32,
                'max_steps': 10000, 'gpu_name': 'CPU', 'vram_gb': 0}

    gpu_name = str(gpu_device)
    print(f'\n🖥️  GPU Device: {gpu_name}')

    # Probe available VRAM
    try:
        stats = gpu_device.memory_stats()
        vram_bytes = stats.get('bytes_limit', 0)
        vram_gb = vram_bytes / (1024**3)
    except Exception:
        import subprocess
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=10)
            vram_mb = int(result.stdout.strip().split('\n')[0])
            vram_gb = vram_mb / 1024
        except Exception:
            vram_gb = 15.0  # Conservative default (T4-class)

    print(f'💾  VRAM: {vram_gb:.1f} GB')

    # Auto-scale based on VRAM
    if vram_gb >= 70:
        cfg = {'batch_size': 64, 'embed_dim': 256, 'base_features': 128,
               'max_steps': 100000, 'gpu_name': 'A100-80GB', 'vram_gb': vram_gb}
        print('🚀  A100 80GB — MAXIMUM config')
    elif vram_gb >= 35:
        cfg = {'batch_size': 48, 'embed_dim': 256, 'base_features': 128,
               'max_steps': 100000, 'gpu_name': 'A100-40GB', 'vram_gb': vram_gb}
        print('🚀  A100 40GB — HIGH config')
    elif vram_gb >= 20:
        cfg = {'batch_size': 32, 'embed_dim': 192, 'base_features': 96,
               'max_steps': 75000, 'gpu_name': 'L4/A10G', 'vram_gb': vram_gb}
        print('⚡  L4/A10G — MEDIUM-HIGH config')
    elif vram_gb >= 14:
        cfg = {'batch_size': 16, 'embed_dim': 128, 'base_features': 64,
               'max_steps': 50000, 'gpu_name': 'T4/V100', 'vram_gb': vram_gb}
        print('⚡  T4/V100 — STANDARD config')
    else:
        cfg = {'batch_size': 8, 'embed_dim': 128, 'base_features': 64,
               'max_steps': 50000, 'gpu_name': f'GPU ({vram_gb:.0f}GB)', 'vram_gb': vram_gb}
        print(f'💡  Smaller GPU ({vram_gb:.0f}GB) — CONSERVATIVE config')

    print(f'\n📋  Training Config:')
    for k, v in cfg.items():
        print(f'    {k:15s} = {v}')
    return cfg

GPU_CONFIG = detect_gpu_and_scale()

# Save for downstream cells
with open('./gpu_config.json', 'w') as f:
    json.dump(GPU_CONFIG, f, indent=2)
print(f'\n✅  Config saved to ./gpu_config.json')

# Verify V2 architecture files
print('\n--- V2 Architecture Check ---')
!grep -l 'LabelEncoder\|EulerMaruyamaSampler\|PhasePool\|predict_meta_physics' sife/model.py sife/diffusion.py sife/unet.py 2>/dev/null

## Cell 3: CIFAR-100 Data Download
Downloads and prepares the CIFAR-100 dataset.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

!python scripts/download_cifar.py

import numpy as np
images = np.load('./data/cifar100/train_images.npy')
labels = np.load('./data/cifar100/train_labels.npy')
print(f'Dataset loaded: {images.shape} images, {labels.shape} labels')
print(f'Image range: [{images.min():.3f}, {images.max():.3f}]')

## Cell 4: GPU-Optimized V2 Training

Training parameters are **automatically scaled** based on the GPU detected in Cell 2.

| GPU Tier  | Batch | Embed Dim | Base Feat | Max Steps |
|-----------|-------|-----------|-----------|-----------|
| A100 80GB | 64    | 256       | 128       | 100,000   |
| A100 40GB | 48    | 256       | 128       | 100,000   |
| L4 / A10G | 32    | 192       | 96        | 75,000    |
| T4 / V100 | 16    | 128       | 64        | 50,000    |
| Smaller   | 8     | 128       | 64        | 50,000    |

**V2 Enhancements:**
- `ImageEncoder` — learned RGB→complex projection
- `LabelEncoder` — CIFAR-100 class labels as complex context
- 15% CFG dropout → class-conditional generation at inference
- `predict_meta_physics()` — dynamic Hamiltonian params
- `PhasePool/PhaseUnpool` — 50% FLOPs reduction

In [ ]:
import os, json
os.environ['PYTHONPATH'] = '.'
os.environ['JAX_TRACEBACK_FILTERING'] = 'off'

# Load GPU-scaled config
with open('./gpu_config.json', 'r') as f:
    gpu_cfg = json.load(f)

print(f"🖥️  Training on: {gpu_cfg['gpu_name']} ({gpu_cfg['vram_gb']:.1f} GB VRAM)")
print(f"📋  batch_size={gpu_cfg['batch_size']}  embed_dim={gpu_cfg['embed_dim']}  "
      f"base_features={gpu_cfg['base_features']}  max_steps={gpu_cfg['max_steps']}")

# Run GPU-optimized training
!python scripts/train_vision_gpu.py \
    --batch_size {gpu_cfg['batch_size']} \
    --embed_dim {gpu_cfg['embed_dim']} \
    --base_features {gpu_cfg['base_features']} \
    --max_steps {gpu_cfg['max_steps']}

## Cell 5: Monitor Training Loss

Expected loss curve:
- Step 0: ~6.0 (random noise)
- Step 1,000: ~2.0–3.5
- Step 10,000: ~1.2–1.8
- Step 50,000: ~0.8–1.2 (good generative quality)

In [ ]:
import matplotlib.pyplot as plt
import re, os

log_file = 'training_log.txt'
if os.path.exists(log_file):
    steps, losses = [], []
    with open(log_file) as f:
        for line in f:
            m = re.search(r'Step\s+(\d+).*Loss\s*=\s*([\d.]+)', line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))

    plt.figure(figsize=(12, 4))
    plt.plot(steps, losses, 'b-', linewidth=1.5, label='Diffusion Loss')
    plt.axhline(y=2.0, color='g', linestyle='--', alpha=0.5, label='Good quality threshold')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.title('SIFE-LDM V2 Training Curve (CIFAR-100)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('No log file found. Training output is inline above.')

## Cell 6: Generate Images (Unconditional)

Samples from pure noise using the `EulerMaruyamaSampler` with attractor-based early stopping.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.'

!python scripts/generate_images.py \
    --checkpoint checkpoints/vision/final_vision_model.pkl \
    --num_images 16 \
    --num_steps 100 \
    --output_dir ./generated_images \
    --attractor_stop

# Display the grid
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('./generated_images/generated_unconditional.png')
plt.figure(figsize=(14, 14))
plt.imshow(img)
plt.axis('off')
plt.title('SIFE-LDM V2 — Unconditional Generation (16 samples)', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 7: Class-Conditional Generation (CFG)

Uses `LabelEncoder` + `cfg_guided_sample()` to steer the SIFE field toward specific CIFAR-100 classes.

**Guidance scale:** Higher = more class-faithful, less variety. 7.5 is a good default.

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image
os.environ['PYTHONPATH'] = '.'

# CIFAR-100 classes to generate
CLASSES_TO_GENERATE = {
    69: 'rocket',
    3:  'bear',
    55: 'otter',
    35: 'girl',
}

fig, axes = plt.subplots(1, len(CLASSES_TO_GENERATE), figsize=(16, 5))

for ax, (class_id, class_name) in zip(axes, CLASSES_TO_GENERATE.items()):
    !python scripts/generate_images.py \
        --checkpoint checkpoints/vision/final_vision_model.pkl \
        --class_id {class_id} \
        --guidance_scale 7.5 \
        --num_images 4 \
        --num_steps 100 \
        --output_dir ./generated_images

    img_path = f'./generated_images/generated_class{class_id}_{class_name}.png'
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f'[{class_id}] {class_name}', fontsize=12)
    ax.axis('off')

fig.suptitle('SIFE-LDM V2 — Class-Conditional CFG Generation (s=7.5)', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 8: Attractor Efficiency Benchmark

Measures how many SDE steps the attractor early-stopping saves — the key SIFE physics advantage.

In [ ]:
import sys, os, pickle, jax, jax.numpy as jnp
os.environ['PYTHONPATH'] = '.'
sys.path.insert(0, '.')

from sife.model import SIFELDM, SIFELDMConfig, LabelEncoder, create_model
from sife.field import SIFEConfig
from sife.diffusion import DiffusionConfig, GaussianDiffusion, EulerMaruyamaSampler
from sife.multiscale import create_multiscale_config

# Load checkpoint
with open('checkpoints/vision/final_vision_model.pkl', 'rb') as f:
    data = pickle.load(f)
params = data.params if hasattr(data, 'params') else data

config = SIFELDMConfig(
    sife=SIFEConfig(), diffusion=DiffusionConfig(num_timesteps=1000),
    multiscale=create_multiscale_config(num_levels=3, base_features=64),
    is_image=True, image_size=(32, 32), embed_dim=128, batch_size=4
)
key = jax.random.PRNGKey(42)
model, _ = create_model(config, key)
diffusion = GaussianDiffusion(DiffusionConfig(num_timesteps=1000))
sampler = EulerMaruyamaSampler(diffusion)

def model_fn(x, t, context=None):
    return model.apply(params, x, t, context=context, deterministic=True)

# Run 5 samples and measure steps saved
shape = (4, 32, 32, 128)
results = []
for trial in range(5):
    key, subkey = jax.random.split(key)
    _, steps_used = sampler.sample(
        model_fn, shape, subkey,
        num_steps=100,
        sife_config=config.sife,
        stability_threshold=0.5
    )
    pct_saved = (100 - steps_used) / 100 * 100
    results.append((steps_used, pct_saved))
    print(f'Trial {trial+1}: {steps_used}/100 steps used → {pct_saved:.1f}% saved')

avg_steps = sum(r[0] for r in results) / len(results)
avg_saved = sum(r[1] for r in results) / len(results)
print(f'\nAverage: {avg_steps:.1f} steps, {avg_saved:.1f}% compute saved via attractor stopping')

## Cell 9: Save Output & Cleanup

In [ ]:
import shutil, os, glob

# Zip all generated images for easy download
shutil.make_archive('sife_v2_generated', 'zip', './generated_images')
print('Images zipped to sife_v2_generated.zip — download from Colab Files panel.')

# Optional: clean up intermediate checkpoints to save disk space
intermediate = glob.glob('checkpoints/vision/checkpoint_*.pkl')
for f in intermediate:
    os.remove(f)
print(f'Cleaned {len(intermediate)} intermediate checkpoints. Final model preserved.')